In [ ]:
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Point to your generated PKL file (002) for a focused sanity check.
pkl_path = Path("/home/wsl2/lijinke/learning/BatteryLife/dataset/processed/Farasis/Farasis_PA-b1_002_single_parquet_with_soc_and_corrections.pkl")

def _get_field(obj, key, default=None):
    """Read field from dict-like or object-like structures."""
    if isinstance(obj, dict):
        return obj.get(key, default)
    return getattr(obj, key, default)

def _cycle_to_dataframe(cycle):
    """Convert one cycle (CycleData object or dict) to a unified DataFrame."""
    current = _get_field(cycle, "current_in_A", [])
    voltage = _get_field(cycle, "voltage_in_V", [])
    charge_capacity = _get_field(cycle, "charge_capacity_in_Ah", [])
    discharge_capacity = _get_field(cycle, "discharge_capacity_in_Ah", [])
    test_time = _get_field(cycle, "time_in_s", [])
    cycle_number = _get_field(cycle, "cycle_number", np.nan)

    min_len = min(
        len(current), len(voltage), len(charge_capacity), len(discharge_capacity), len(test_time)
    )
    if min_len == 0:
        return pd.DataFrame()

    df = pd.DataFrame({
        "current": np.asarray(current[:min_len], dtype=float),
        "voltage": np.asarray(voltage[:min_len], dtype=float),
        "charge_capacity": np.asarray(charge_capacity[:min_len], dtype=float),
        "discharge_capacity": np.asarray(discharge_capacity[:min_len], dtype=float),
        "test_time": np.asarray(test_time[:min_len], dtype=float),
        "cycle_number": cycle_number,
    })
    return df

# Load PKL that may be BatteryData object or plain dict.
with open(pkl_path, "rb") as f:
    cell_data = pickle.load(f)

cycle_data = _get_field(cell_data, "cycle_data", [])
nominal_capacity = _get_field(cell_data, "nominal_capacity_in_Ah", np.nan)
cell_id = _get_field(cell_data, "cell_id", pkl_path.stem)

if not cycle_data:
    raise ValueError(f"No cycle_data found in {pkl_path}")

# Build concatenated trajectory for plotting and per-cycle max capacity checks.
all_frames = []
cycle_stats = []

for cycle in cycle_data:
    cycle_df = _cycle_to_dataframe(cycle)
    if cycle_df.empty:
        continue

    all_frames.append(cycle_df)

    dis_max = cycle_df.loc[cycle_df["current"] < 0, "discharge_capacity"].max()
    chg_max = cycle_df.loc[cycle_df["current"] > 0, "charge_capacity"].max()
    cycle_no = int(_get_field(cycle, "cycle_number", len(cycle_stats) + 1))
    cycle_stats.append((cycle_no, chg_max, dis_max))

if not all_frames:
    raise ValueError("All cycles are empty after parsing; please check pkl schema.")

plot_df = pd.concat(all_frames, ignore_index=True)

fig, ax1 = plt.subplots(figsize=(12, 6))
ax1.plot(plot_df["test_time"], plot_df["charge_capacity"], color="tab:blue", linewidth=1, label="Charge capacity")
ax1.set_xlabel("Time (s)")
ax1.set_ylabel("Charge Capacity (Ah)", color="tab:blue")
ax1.tick_params(axis="y", labelcolor="tab:blue")

ax2 = ax1.twinx()
ax2.plot(plot_df["test_time"], plot_df["discharge_capacity"], color="tab:red", linewidth=1, label="Discharge capacity")
ax2.set_ylabel("Discharge Capacity (Ah)", color="tab:red")
ax2.tick_params(axis="y", labelcolor="tab:red")

plt.title(f"Capacity vs Time | {cell_id} | nominal={nominal_capacity} Ah")
plt.tight_layout()
plt.show()

# Print first cycles summary to quickly verify magnitude is reasonable.
stats_df = pd.DataFrame(cycle_stats, columns=["cycle_number", "charge_max_Ah", "discharge_max_Ah"])
print(stats_df.head(20))
print(f"Parsed cycles: {len(stats_df)}")
print(f"Discharge max over all cycles: {stats_df['discharge_max_Ah'].max():.3f} Ah")